# SAFE-Alert — Train BTCUSDT 1h

Notebook độc lập cho Kaggle/Colab. Dữ liệu 1h dùng 16 news candidates trong cửa sổ 24 giờ; model chọn Top-4. Output được lưu tại `/kaggle/working/safe_alert_1h`.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess

HORIZON = '1h'
SYMBOL = 'BTCUSDT'
EPOCHS = 40
N_FOLDS = 5
BATCH_SIZE = 8
RUN_SMOKE_TEST = True

candidates = [Path.cwd(), Path('/kaggle/working/CQ_2025-AI_Sentiment_Support-Private/services/ai-service')]
if Path('/kaggle/input').exists(): candidates += [p.parents[3] for p in Path('/kaggle/input').glob('**/app/v2/pipelines/train_safe_alert.py')]
SERVICE_ROOT = next((p.resolve() for p in candidates if (p/'app/v2/pipelines/train_safe_alert.py').exists()), None)
assert SERVICE_ROOT, 'Không tìm thấy services/ai-service. Hãy add repository/dataset vào notebook.'
DATA = SERVICE_ROOT/'training_data/v2'
CONFIG = SERVICE_ROOT/'app/v2/pipelines/train_config_research_best.yaml'
ARTIFACTS = Path('/kaggle/working/safe_alert_1h') if Path('/kaggle/working').exists() else SERVICE_ROOT/'artifacts/safe_alert_1h'
print('SERVICE_ROOT =', SERVICE_ROOT)
print('DATA         =', DATA)
print('ARTIFACTS    =', ARTIFACTS)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ta==0.11.0', 'vaderSentiment==3.3.2', 'PyYAML>=6.0.3,<6.1'], check=True)
import numpy as np, pandas as pd, torch, yaml
print('torch=', torch.__version__, 'cuda=', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Nên bật GPU accelerator trước khi full train.'

In [ ]:
required = ['BTCUSDT_1h_ohlcv.csv','articles_max.csv','btcusdt_article_embeddings_max.npy','features_precomputed.npy','features_precomputed.meta.json','market_bars.npz','article_factor_labels.npy','article_entity_sentiment.npy','article_novelty.npy']
missing = [n for n in required if not (DATA/n).exists()]
assert not missing, f'Thiếu file: {missing}'
import csv
def count_csv_records(path):
    with path.open(encoding='utf-8-sig', errors='replace', newline='') as f:
        reader=csv.reader(f); next(reader)
        return sum(1 for row in reader if row)
n_candles = count_csv_records(DATA/'BTCUSDT_1h_ohlcv.csv')
n_articles = count_csv_records(DATA/'articles_max.csv')
checks = {
 'features': np.load(DATA/'features_precomputed.npy', mmap_mode='r').shape,
 'embeddings': np.load(DATA/'btcusdt_article_embeddings_max.npy', mmap_mode='r').shape,
 'factors': np.load(DATA/'article_factor_labels.npy', mmap_mode='r').shape,
 'entity_sentiment': np.load(DATA/'article_entity_sentiment.npy', mmap_mode='r').shape,
 'novelty': np.load(DATA/'article_novelty.npy', mmap_mode='r').shape,
}
with np.load(DATA/'market_bars.npz') as z: bar_shapes = {k:z[k].shape for k in z.files}
assert checks['features'] == (n_candles, 63)
assert checks['embeddings'] == (n_articles, 768)
assert checks['factors'] == checks['entity_sentiment'] == (n_articles, 10)
assert checks['novelty'] == (n_articles,)
assert all(s == (n_candles, 20, 10) for s in bar_shapes.values())
print({'candles':n_candles, 'articles':n_articles, **checks, 'bars':bar_shapes})

In [ ]:
cmd_base = [sys.executable, 'app/v2/pipelines/train_safe_alert.py', '--config', str(CONFIG), '--symbol', SYMBOL, '--horizon', HORIZON, '--data_path', str(DATA), '--embeddings_path', str(DATA), '--batch_size', str(BATCH_SIZE), '--walk_forward', '--use_bar_sequences']
if RUN_SMOKE_TEST:
    smoke = ARTIFACTS.parent/'smoke_1h'
    subprocess.run(cmd_base + ['--epochs','2','--n_folds','1','--artifact_dir',str(smoke)], cwd=SERVICE_ROOT, check=True)
print('Smoke test hoàn tất.')

In [ ]:
ARTIFACTS.mkdir(parents=True, exist_ok=True)
cmd = cmd_base + ['--epochs',str(EPOCHS),'--n_folds',str(N_FOLDS),'--artifact_dir',str(ARTIFACTS)]
print(' '.join(map(str,cmd)))
subprocess.run(cmd, cwd=SERVICE_ROOT, check=True)

In [ ]:
zip_path = shutil.make_archive(str(ARTIFACTS), 'zip', ARTIFACTS)
print('Download artifact:', zip_path)
print('\n'.join(str(p.relative_to(ARTIFACTS)) for p in sorted(ARTIFACTS.rglob('*')) if p.is_file()))